In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data.loader import download_prices, download_volume, download_fundamentals
from data.universe import get_sp500_tickers, build_return_matrix
from factors.momentum import MomentumFactor, ReversalFactor
from factors.volatility import IVolFactor
from factors.illiquidity import IlliquidityFactor
from factors.earnings_yield import EarningsYieldFactor
from evaluation.ic import compute_ic_series, compute_ic_stats
from evaluation.quintile import compute_quintile_returns, compute_quintile_stats
from evaluation.decay import compute_factor_decay, compute_factor_correlation
from portfolio.combination import equal_weight_alpha, ic_weighted_alpha
from portfolio.backtest import run_backtest, compute_performance_metrics
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
tickers = get_sp500_tickers()
prices = download_prices(tickers, "2009-01-01", "2024-12-31")
volume = download_volume(tickers, "2009-01-01", "2024-12-31")
fundamentals = download_fundamentals(tickers)

import yfinance as yf
spy_daily = yf.download("SPY", start="2009-01-01", end="2024-12-31", auto_adjust=True, progress=False)["Close"].to_frame()
spy_monthly_ret = spy_daily.resample("ME").last().loc["2010":"2024"].iloc[:, 0].pct_change()

monthly_closes, monthly_returns = build_return_matrix(prices, "2010-01-01", "2024-12-31")
rebalance_dates = monthly_closes.index
forward_returns = monthly_returns.shift(-1)
IS_END = "2019-12-31"

In [ ]:
factors = {
    "MOM":   MomentumFactor().compute(prices, volume, None, rebalance_dates),
    "REV":   ReversalFactor().compute(prices, volume, None, rebalance_dates),
    "IVOL":  IVolFactor(spy_prices=spy_daily).compute(prices, volume, None, rebalance_dates),
    "ILLIQ": IlliquidityFactor().compute(prices, volume, None, rebalance_dates),
    "EP":    EarningsYieldFactor().compute(prices, volume, fundamentals, rebalance_dates),
}
print({k: f.notna().sum(axis=1).mean() for k, f in factors.items()})

In [ ]:
rows = []
for name, f in factors.items():
    for period, start, end in [("IS", "2010-01-01", IS_END), ("OOS", "2020-01-01", "2024-12-31")]:
        ic = compute_ic_series(f.loc[start:end], forward_returns.loc[start:end])
        s = compute_ic_stats(ic)
        rows.append({"Factor": name, "Period": period, **{k: round(v, 4) for k, v in s.items()}})
pd.DataFrame(rows).pivot_table(index="Factor", columns="Period", values=["mean_ic", "icir", "t_stat"])

In [ ]:
fig, axes = plt.subplots(1, len(factors), figsize=(18, 4), sharey=False)
for ax, (name, f) in zip(axes, factors.items()):
    q_ret = compute_quintile_returns(f.loc[:IS_END], forward_returns.loc[:IS_END])
    stats = compute_quintile_stats(q_ret)
    stats["ann_return"].plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(name)
    ax.set_xlabel("")
    ax.axhline(0, color="black", linewidth=0.8)
plt.suptitle("Annualized Return by Quintile (In-Sample 2010\u20132019)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(factors), figsize=(18, 4), sharey=True)
horizons = [1, 2, 3, 6, 12]
extra_returns = pd.concat([forward_returns,
    pd.DataFrame(0.0, index=pd.date_range(forward_returns.index[-1] + pd.offsets.MonthEnd(1),
                 periods=12, freq="ME"), columns=forward_returns.columns)])
for ax, (name, f) in zip(axes, factors.items()):
    decay = compute_factor_decay(f.loc[:IS_END], extra_returns.loc[:IS_END], horizons)
    ax.plot(decay.index, decay["mean_ic"], marker="o")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(name); ax.set_xlabel("Horizon (months)")
axes[0].set_ylabel("Mean IC")
plt.suptitle("Factor Decay: Mean IC by Forward Return Horizon (In-Sample)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
corr = compute_factor_correlation(factors)
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Average Cross-Sectional Factor Correlations (Full Sample)")
plt.tight_layout(); plt.show()

In [ ]:
ew_alpha = equal_weight_alpha(factors)
ic_alpha = ic_weighted_alpha(factors, forward_returns)

for label, alpha in [("Equal-Weight", ew_alpha), ("IC-Weighted", ic_alpha)]:
    result = run_backtest(alpha, monthly_returns, spy_returns=spy_monthly_ret)
    print(f"\n{label} \u2014 In-Sample Gross / Net:")
    for k, m in [("Gross", result["metrics_gross"]), ("Net", result["metrics_net"])]:
        r = result["returns"]["gross" if k == "Gross" else "net"].loc[:IS_END]
        m_is = compute_performance_metrics(r, spy_monthly_ret.loc[:IS_END])
        print(f"  {k}: Sharpe={m_is['sharpe_ratio']:.2f}, "
              f"Ann Ret={m_is['annualized_return']:.1%}, "
              f"Max DD={m_is['max_drawdown']:.1%}, "
              f"Beta={m_is['beta_to_spy']:.2f}")

In [ ]:
ew_result = run_backtest(ew_alpha, monthly_returns, spy_returns=spy_monthly_ret)
ic_result = run_backtest(ic_alpha, monthly_returns, spy_returns=spy_monthly_ret)

fig, ax = plt.subplots(figsize=(12, 5))
for label, result in [("Equal-Weight (Net)", ew_result), ("IC-Weighted (Net)", ic_result)]:
    cum = (1 + result["returns"]["net"]).cumprod()
    cum.plot(ax=ax, label=label)
ax.axvline(pd.Timestamp(IS_END), color="red", linestyle="--", label="OOS Start")
ax.set_title("Long-Short Cumulative NAV")
ax.set_ylabel("Cumulative Return"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
rows = []
for label, result in [("Equal-Weight", ew_result), ("IC-Weighted", ic_result)]:
    for period_label, start, end in [("In-Sample 2010\u20132019", "2010-01-01", IS_END),
                                      ("Out-of-Sample 2020\u20132024", "2020-01-01", "2024-12-31")]:
        for cost_label, col in [("Gross", "gross"), ("Net", "net")]:
            r = result["returns"][col].loc[start:end]
            m = compute_performance_metrics(r, spy_monthly_ret.loc[start:end])
            rows.append({"Strategy": label, "Period": period_label, "Cost": cost_label,
                         "Sharpe": round(m["sharpe_ratio"], 2),
                         "Ann Return": f"{m['annualized_return']:.1%}",
                         "Max DD": f"{m['max_drawdown']:.1%}",
                         "Beta": round(m["beta_to_spy"], 2)})
pd.DataFrame(rows)